In [1]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer

C:\Users\navid.hejazi\AppData\Local\anaconda3\envs\langchain_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
files = pd.read_csv("course_descriptions.csv", encoding="ANSI")

In [3]:
files.head()

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a..."
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...


In [4]:
def create_course_description(row):
    return f""" The course name is {row["course_name"]}, the slug is {row["course_slug"]}, 
    The technology is {row["course_technology"]}, and the course topic is {row["course_topic"]}
    """

In [5]:
files["course_description_new"] = files.apply(create_course_description, axis=1 )
print(files["course_description_new"])

0       The course name is Introduction to Tableau, t...
1       The course name is The Complete Data Visualiz...
2       The course name is Introduction to R Programm...
3       The course name is Data Preprocessing with Nu...
4       The course name is Introduction to Data and D...
                             ...                        
101     The course name is Intro to NLP for AI, the s...
102     The course name is Data Analysis with ChatGPT...
103     The course name is ChatGPT for Data Science, ...
104     The course name is Intro to LLMs, the slug is...
105     The course name is Growth Analysis with SQL, ...
Name: course_description_new, Length: 106, dtype: object


In [3]:
%load_ext dotenv
%dotenv

In [4]:
load_dotenv(find_dotenv(), override = True)

True

In [5]:
pc=Pinecone(api_key = os.environ.get("PINECONE_API_KEY"), environment = os.environ.get("PINECONE_ENV"))

In [6]:
index_name = "my-index"
dimension = 384
metric = "cosine"

In [12]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")

my-index not in index list.


In [13]:
pc.create_index(
    name = index_name, 
    dimension = dimension, 
    metric = metric, 
    spec = ServerlessSpec(
        cloud = "aws", 
        region = "us-east-1")
    )

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-1bs5h5j.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "access-control-allow-origin": "*",
            "vary": "origin,access-control-request-method,access-control-request-headers",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
  

In [7]:
index = pc.Index(index_name)

### Embedding the data


In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [19]:
def create_embeddings(row):
    combined_text = ' '.join([str(row[field]) for field in ["course_description", "course_description_new", "course_description_short"]])
    embedding = model.encode(combined_text, show_progress_bar = False)
    return embedding

In [20]:
files["embeddings"] = files.apply(create_embeddings, axis=1)

In [24]:
vectors_to_upsert = [(str(row["course_name"]), row["embeddings"].tolist()) for _,row in files.iterrows()]
index.upsert(vectors=vectors_to_upsert)

UpsertResponse(upserted_count=106, _response_info={'raw_headers': {'date': 'Fri, 09 Jan 2026 15:18:42 GMT', 'content-type': 'application/json', 'content-length': '21', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '165995', 'x-pinecone-request-latency-ms': '794', 'x-pinecone-request-id': '4927784494131837715', 'x-envoy-upstream-service-time': '317', 'x-pinecone-response-duration-ms': '796', 'grpc-status': '0', 'server': 'envoy'}})

### Semantic search

In [9]:
query = "regression in python"
query_embedding = model.encode(query, show_progress_bar = False).tolist()

In [10]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12,
    include_values = True
)

In [11]:
query_results

QueryResponse(matches=[{'id': 'Machine Learning in Python',
 'score': 0.52942282,
 'values': [-0.0439037755,
            -0.0421731062,
            -0.0108734462,
            0.0417844951,
            -0.0196499396,
            -0.115790457,
            -0.0199701432,
            -0.0508757271,
            -0.110716447,
            -0.0282511711,
            -0.0713698193,
            0.0381572135,
            0.0699843392,
            0.0201659985,
            0.0253228787,
            -0.00949908,
            -0.0545141809,
            0.000200288021,
            0.0228322633,
            -0.109917656,
            0.0272190068,
            0.0492703207,
            -0.0258642845,
            0.0231282711,
            -0.0217068195,
            -0.0276037473,
            0.0686967596,
            0.050078731,
            -0.0921665877,
            0.000897411082,
            -0.000767933961,
            0.0150447795,
            0.00168191968,
            0.0212773923,
            -0.

In [12]:
for match in query_results["matches"]:
    print(f"Matched item ID: {match['id']}, score:{match['score']}")

Matched item ID: Machine Learning in Python, score:0.52942282
Matched item ID: Introduction to Python, score:0.462595969
Matched item ID: Machine Learning with Ridge and Lasso Regression, score:0.461896956
Matched item ID: Python for Finance, score:0.441794455
Matched item ID: Customer Analytics in Python, score:0.433410674
Matched item ID: Data Preprocessing with NumPy, score:0.426073104
Matched item ID: Credit Risk Modeling in Python, score:0.407000601
Matched item ID: Intermediate Python Programming, score:0.395573676
Matched item ID: Introduction to Jupyter, score:0.386819899
Matched item ID: Time Series Analysis with Python, score:0.382012367
Matched item ID: Data Cleaning and Preprocessing with pandas, score:0.371097594
Matched item ID: Machine Learning with Support Vector Machines, score:0.370792419


In [13]:
score_threshold = 0.3
for match in query_results["matches"]:
    if match["score"] >= score_threshold:
        print(f"Matched item ID: {match['id']}, score:{match['score']}")

Matched item ID: Machine Learning in Python, score:0.52942282
Matched item ID: Introduction to Python, score:0.462595969
Matched item ID: Machine Learning with Ridge and Lasso Regression, score:0.461896956
Matched item ID: Python for Finance, score:0.441794455
Matched item ID: Customer Analytics in Python, score:0.433410674
Matched item ID: Data Preprocessing with NumPy, score:0.426073104
Matched item ID: Credit Risk Modeling in Python, score:0.407000601
Matched item ID: Intermediate Python Programming, score:0.395573676
Matched item ID: Introduction to Jupyter, score:0.386819899
Matched item ID: Time Series Analysis with Python, score:0.382012367
Matched item ID: Data Cleaning and Preprocessing with pandas, score:0.371097594
Matched item ID: Machine Learning with Support Vector Machines, score:0.370792419
